In [1]:
import json
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/seminar/

/content/drive/MyDrive/seminar


In [4]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch
import os
from IPython.display import display

# Model ID
model_id = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Optional but important: set patch size if missing
if hasattr(processor.image_processor, "patch_size") and processor.image_processor.patch_size is None:
    processor.image_processor.patch_size = 14
else:
    processor.patch_size = 14

# Load image
#image_path = "/content/drive/MyDrive/seminar/0cbaca12e05803e6301f8f4a92b47565.png"
images_path ="/content/drive/MyDrive/seminar/nova_brain/images"
results = []



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [5]:
for image_name in os.listdir(images_path):
  print(image_name)
  image_path = os.path.join(images_path, image_name)
  image = Image.open(image_path).convert("RGB")

  # visualize the image
  #display(image)

  prompt = """USER: <image>\n You are a brain MRI medical assistant. Carefully analyze and describe the image. The description should be in the form of a caption with the most important findings. ASSISTANT:"""

  # Prepare inputs (text prompt + image)
  inputs = processor(
      text=prompt,
      images=image,
      return_tensors="pt"
  )

  #print("inputs:", inputs)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  # Run inference
  with torch.no_grad():
      generated_ids = model.generate(
          **inputs,
          max_new_tokens=128,
          min_new_tokens=1,
          do_sample=True,
          temperature=0.7,
          top_p = 0.8,
          pad_token_id=processor.tokenizer.pad_token_id,
          eos_token_id=processor.tokenizer.eos_token_id,
          use_cache=True
      )

  # Decode result
  full_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

  # Extract only the assistant's part
  if "ASSISTANT:" in full_response:
      assistant_response = full_response.split("ASSISTANT:")[-1].strip()
  else:
      assistant_response = full_response

  #print("Full response:", full_response)
  print("Assistant response:", assistant_response)

  results.append({
            'image': image_name,
            'prediction': assistant_response
        })

case0005_002.png
Assistant response: The MRI image of the brain shows several findings, including dilated ventricles, diffuse atrophy, and cerebellar atrophy. These findings suggest that there may be some underlying neurological issues or conditions affecting the brain. Further evaluation and clinical correlation are needed to determine the cause and significance of these findings.
case0070_008.png
Assistant response: The MRI image shows multiple lesions in the brain, including a large lesion in the left parietal lobe. These lesions may be indicative of an underlying neurological condition or disease. Further evaluation and consultation with a healthcare professional are necessary to determine the cause and appropriate treatment for these findings.
case0127_002.png
Assistant response: The brain MRI image shows the presence of bilateral symmetrical high signal intensity in the globus pallidus internus, as well as a high signal intensity in the thalamus. These findings may indicate an un

In [6]:
import csv

#save all results to a csv file with 2 columns - image and prediction
csv_output_path = "/content/drive/MyDrive/seminar/mri_descriptions.csv"

with open(csv_output_path, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image', 'prediction']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header
    writer.writeheader()

    # Write data
    for result in results:
        writer.writerow(result)

In [8]:
results = []

for image_name in os.listdir(images_path):
    image_path = os.path.join(images_path, image_name)
    image = Image.open(image_path).convert("RGB")
    prompt = f"""USER: <image>\n Carefully analyze the provided brain MRI and locate the abnormal region. Provide ONLY the bounding box coordinates for the abnormal region in the following format: (x1, y1, x2, y2). ASSISTANT:"""
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )

    #print("inputs:", inputs)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Run inference
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50, # Increased max_new_tokens to allow for full coordinate output
            min_new_tokens=1,
            do_sample=True,
            temperature=0.7,
            top_p = 0.8,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True
        )

    # Decode result
    full_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Extract only the assistant's part
    if "ASSISTANT:" in full_response:
        assistant_response = full_response.split("ASSISTANT:")[-1].strip()
    else:
        assistant_response = full_response

    #print("Full response:", full_response)
    print("Assistant response:", assistant_response)

    results.append({
              'image': image_name,
              'prediction': assistant_response
          })

Assistant response: The abnormal region is located in the right frontal lobe. The bounding box coordinates for this region are (31, 17, 42, 30).
Assistant response: The abnormal region is located at (15, 25, 22, 33).
Assistant response: The abnormal region in the brain MRI is located at (34, 21, 40, 30).
Assistant response: The abnormal region in the brain MRI is located at (35, 27, 38, 26).
Assistant response: The abnormal region is located in the left cerebellar hemisphere. The coordinates for this region are (21, 20, 28, 26).
Assistant response: The abnormal region in the brain MRI is located at (55, 26, 60, 32).
Assistant response: The abnormal region is located at (47, 10, 57, 40).
Assistant response: The abnormal region is located in the right temporal lobe. The bounding box coordinates for this region are (20, 20, 40, 40).
Assistant response: The abnormal region is located at (15, 12, 16, 11).
Assistant response: The abnormal region is located in the left occipital lobe. The bou

In [9]:
import csv
csv_output_path = "/content/drive/MyDrive/seminar/mri_predictions_grounding.csv"

with open(csv_output_path, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image', 'prediction']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header
    writer.writeheader()

    # Write data
    for result in results:
        writer.writerow(result)